<a href="https://colab.research.google.com/github/pranathi139/GenAI/blob/main/Week3_Invocation_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install requests beautifulsoup4 transformers sentencepiece


In [ ]:
import requests
from bs4 import BeautifulSoup
from transformers import pipeline

In [ ]:
class SimpleMemory:
    def __init__(self):
        self.history = []

    def add(self, query, result):
        self.history.append({"query": query, "result": result})

    def get_all(self):
        return self.history

    def show(self):
        for i, item in enumerate(self.history):
            print(f"\n[{i+1}] QUERY: {item['query']}")
            print(f"RESULT: {item['result'][:200]}...")


In [ ]:
def fetch_web_content(url):
    try:
        response = requests.get(url, timeout=10)
        soup = BeautifulSoup(response.text, "html.parser")

        # Extract paragraph text
        paragraphs = [p.get_text() for p in soup.find_all("p")]
        text = " ".join(paragraphs)

        return text[:5000]  # limit size
    except Exception as e:
        return f"Error fetching content: {e}"

In [ ]:
!pip uninstall -y transformers huggingface_hub tokenizers accelerate
!pip install transformers==4.40.2 huggingface_hub==0.23.0 tokenizers==0.19.1 accelerate sentencepiece

In [ ]:
from transformers import pipeline


summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

def summarize_text(text):
    if len(text) < 100:
        return "Text too short to summarize."

    summary = summarizer(text[:1000], max_length=130, min_length=30, do_sample=False)
    return summary[0]['summary_text']


# Test
text = """Artificial intelligence (AI) is intelligence demonstrated by machines,
unlike the natural intelligence displayed by humans and animals."""

print(summarize_text(text))


In [ ]:
import requests
from bs4 import BeautifulSoup


# ✅ Tool 1: Fetch Web Content
def fetch_web_content(url):
    try:
        response = requests.get(url)
        soup = BeautifulSoup(response.text, "html.parser")

        paragraphs = soup.find_all("p")
        text = " ".join([p.get_text() for p in paragraphs])

        return text[:2000]
    except Exception as e:
        return f"Error fetching data: {e}"


# ✅ Memory Class
class Memory:
    def __init__(self):
        self.history = []

    def add(self, query, result):
        self.history.append({"query": query, "result": result})

    def show(self):
        print("\n--- MEMORY ---")
        for i, item in enumerate(self.history):
            print(f"\n{i+1}. URL: {item['query']}")
            print(f"Summary: {item['result'][:150]}...")

In [ ]:
class Agent:
    def __init__(self):
        self.memory = Memory()

    def run(self, url):
        print("Step 1: Fetching content...")
        content = fetch_web_content(url)

        print("Step 2: Summarizing...")
        summary = summarize_text(content)

        self.memory.add(url, summary)

        return summary

In [ ]:
agent = Agent()

url = "https://www.geeksforgeeks.org/deep-learning/neural-networks-a-beginners-guide/"

result = agent.run(url)

print("\n✅ FINAL SUMMARY:\n")
print(result)

In [ ]:
agent.memory.show()

In [ ]:
while True:
    user_input = input("\nEnter URL (or type 'exit'): ")

    if user_input.lower() == "exit":
        break

    result = agent.run(user_input)

    print("\nSummary:\n", result)